<a href="https://colab.research.google.com/github/pejmanrasti/Big_Data/blob/main/04_pyspark_movie_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PySpark MovieLens


## 0. Setup (Run this cell)

In [ ]:
!pip install -q pyspark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PySpark_Student_Exercises_Movies")
    .master("local[*]")
    .getOrCreate()
)

spark

## 1. Download & Load MovieLens Dataset

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q ml-latest-small.zip

ratings = spark.read.csv("ml-latest-small/ratings.csv", header=True, inferSchema=True)
movies  = spark.read.csv("ml-latest-small/movies.csv", header=True, inferSchema=True)

replace ml-latest-small/links.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

## Task 1 — Data Exploration

**1.1 Inspect the data**

*	Show first 10 rows of ratings
*	Print schema for both DataFrames
*	Count the number of rows in each




**1.2 Column selection**

Write PySpark code to:

* select only userId, movieId, rating
*	rename movieId → film_id
*	cast rating to integer or float explicitly

**1.3 Filtering**

Using ratings:

*	Find all ratings from userId = 1
*	Find all ratings greater than 4.5
*	Find all ratings on movieId in (1, 50, 100)


(use isin())

**1.4 Sorting & limiting**

*	Show the top 20 highest ratings
*	Show the 10 lowest ratings made by user 600
*	Sort by rating desc and timestamp asc

**1.5 Derived columns**

Create columns:

*	rating_x2 = rating * 2
*	positive_rating = 1 if rating ≥ 4 else 0
*	log_rating = log10(rating + 1)

**1.6 Missing values**

Even if ML-latest-small has no nulls, :

*	Add a fake null column
*	Demonstrate fill/replace/drop operations

**1.7 Distinct & deduplication**

*	Count unique users
*	Count unique movies rated
*	Drop duplicate ratings where (userId, movieId) might repeat (even though dataset is clean)

In [ ]:
print("Ratings DataFrame - First 10 rows:")
ratings.show(10)
print("\nMovies DataFrame - First 10 rows:")
movies.show(10)
print("\nRatings DataFrame Schema:")
ratings.printSchema()
print("\nMovies DataFrame Schema:")
movies.printSchema()
print(f"\nNumber of rows in ratings DataFrame: {ratings.count()}")
print(f"Number of rows in movies DataFrame: {movies.count()}")


# 1.2 Column selection
from pyspark.sql.functions import col
# Select, rename, and cast columns
ratings_selected = ratings.select(col("userId"), col("movieId").alias("film_id"), col("rating").cast("float"))
print("\nRatings DataFrame after column selection, renaming, and casting:")
ratings_selected.show(5)
ratings_selected.printSchema()

# 1.3 Filtering
print("\nRatings from userId = 1:")
ratings.filter(ratings.userId == 1).show(5)
print("\nRatings greater than 4.5:")
ratings.filter(ratings.rating > 4.5).show(5)
print("\nRatings on movieId in (1, 50, 100):")
ratings.filter(ratings.movieId.isin([1, 50, 100])).show(5)

# 1.4 Sorting & limiting
print("\nTop 20 highest ratings:")
ratings.orderBy(col("rating").desc()).show(20)
print("\n10 lowest ratings made by user 600:")
ratings.filter(ratings.userId == 600).orderBy(col("rating").asc()).show(10)
print("\nRatings sorted by rating desc and timestamp asc:")
ratings.orderBy(col("rating").desc(), col("timestamp").asc()).show(10)


#1-5
from pyspark.sql.functions import when, log10
ratings_with_derived = ratings.select(
    "*",
    (col("rating") * 2).alias("rating_x2"),
    when(col("rating") >= 4, 1).otherwise(0).alias("positive_rating"),
    log10(col("rating") + 1).alias("log_rating")
)
print("Ratings with derived columns:")
ratings_with_derived.show(10)

#1.6
from pyspark.sql.functions import lit
from pyspark.sql.types import StringType # Import StringType for explicit casting
ratings_with_nulls = ratings.withColumn("fake_null_col", lit(None).cast(StringType())) # cast to avoid none
print("DataFrame with fake null column:")
ratings_with_nulls.show(5)
filled = ratings_with_nulls.fillna({"fake_null_col": "default_value"})
print("\nAfter filling nulls in fake_null_col:")
filled.show(5)
replaced = ratings.withColumn("modified_rating",
                            when(col("rating") == 5.0, 10.0).otherwise(col("rating")))
print("\nAfter replacing 5.0 with 10.0:")
replaced.filter(col("modified_rating") == 10.0).show(5)
dropped = ratings_with_nulls.dropna(subset=["fake_null_col"])
print(f"\nAfter dropping rows where fake_null_col is null: {dropped.count()} rows")


#1-7
# Count unique users
unique_users = ratings.select("userId").distinct().count()
print(f"Number of unique users: {unique_users}")
unique_movies_rated = ratings.select("movieId").distinct().count()
print(f"Number of unique movies rated: {unique_movies_rated}")
unique_movies_total = movies.select("movieId").distinct().count()
print(f"Number of unique movies in movies dataset: {unique_movies_total}")
duplicate_count = ratings.groupBy("userId", "movieId") \
                         .count() \
                         .filter(col("count") > 1) \
                         .count()
print(f"\nNumber of duplicate (userId, movieId) pairs: {duplicate_count}")
deduplicated = ratings.dropDuplicates(["userId", "movieId"])
print(f"Rows after deduplication: {deduplicated.count()} (original: {ratings.count()})")

Ratings DataFrame - First 10 rows:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|     70|   3.0|964982400|
|     1|    101|   5.0|964980868|
|     1|    110|   4.0|964982176|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
+------+-------+------+---------+
only showing top 10 rows

Movies DataFrame - First 10 rows:
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|    

## Task 2 — Aggregations & GroupBy

**2.1 Simple aggregations**

*	Compute min, max, avg rating
*	Count total number of ratings
*	Count ratings per movieId

**2.2 GroupBy**

Compute:

*	average rating per movie
*	number of ratings per movie
*	average rating per user
*	number of ratings per user

**2.3 Top items**

*	Top 20 most-rated movies
*	Top 20 best-rated movies (min 50 ratings)

→ You must filter with a join or window

**2.4 Window functions**

Using Window partitioned by movieId:

*	rank users by timestamp (earliest → latest rating)
*	create a lag column: previous rating by same user
*	compute average rating per movie with window


In [ ]:
from pyspark.sql.functions import col, min, max, avg, count
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lag, avg as avg_f

# === 2.1 Simple aggregations ===
print("=== 2.1 Simple aggregations ===")
ratings.agg(
    min("rating").alias("min_rating"),
    max("rating").alias("max_rating"),
    avg("rating").alias("avg_rating"),
).show()

print("Total number of ratings:", ratings.count())

ratings_per_movie = ratings.groupBy("movieId").count().withColumnRenamed("count", "n_ratings")
print("Ratings per movieId (sample):")
ratings_per_movie.show(5)

# === 2.2 GroupBy ===
print("=== 2.2 GroupBy ===")
movie_stats = ratings.groupBy("movieId").agg(
    count("*").alias("n_ratings"),
    avg("rating").alias("avg_rating"),
)
print("Average rating and number of ratings per movie (sample):")
movie_stats.show(5)

user_stats = ratings.groupBy("userId").agg(
    count("*").alias("n_ratings"),
    avg("rating").alias("avg_rating"),
)
print("Average rating and number of ratings per user (sample):")
user_stats.show(5)

# === 2.3 Top items ===
print("=== 2.3 Top items ===")
print("Top 20 most-rated movies (by movieId):")
top_most_rated = movie_stats.orderBy(col("n_ratings").desc())
top_most_rated.show(20)

print("Top 20 best-rated movies (min 50 ratings):")
top_best_rated = movie_stats.filter(col("n_ratings") >= 50).orderBy(col("avg_rating").desc())
top_best_rated.show(20)

print("Top 20 best-rated movies with titles (min 50 ratings):")
top_best_rated_with_titles = top_best_rated.join(movies, "movieId")
top_best_rated_with_titles.select("movieId", "title", "n_ratings", "avg_rating").show(20, truncate=False)

# === 2.4 Window functions ===
print("=== 2.4 Window functions ===")

w_movie_time = Window.partitionBy("movieId").orderBy("timestamp")
w_user_time = Window.partitionBy("userId").orderBy("timestamp")
w_movie = Window.partitionBy("movieId")

ratings_with_windows = (
    ratings
    .withColumn("rank_in_movie_by_time", row_number().over(w_movie_time))
    .withColumn("prev_rating_same_user", lag("rating").over(w_user_time))
    .withColumn("avg_movie_rating_window", avg_f("rating").over(w_movie))
)

print("Ratings with window columns (sample):")
ratings_with_windows.select(
    "userId",
    "movieId",
    "rating",
    "timestamp",
    "rank_in_movie_by_time",
    "prev_rating_same_user",
    "avg_movie_rating_window",
).show(20)

=== 2.1 Simple aggregations ===
+----------+----------+-----------------+
|min_rating|max_rating|       avg_rating|
+----------+----------+-----------------+
|       0.5|       5.0|3.501556983616962|
+----------+----------+-----------------+

Total number of ratings: 100836
Ratings per movieId (sample):
+-------+---------+
|movieId|n_ratings|
+-------+---------+
|   1580|      165|
|   2366|       25|
|   3175|       75|
|   1088|       42|
|  32460|        4|
+-------+---------+
only showing top 5 rows
=== 2.2 GroupBy ===
Average rating and number of ratings per movie (sample):
+-------+---------+-----------------+
|movieId|n_ratings|       avg_rating|
+-------+---------+-----------------+
|   1580|      165|3.487878787878788|
|   2366|       25|             3.64|
|   3175|       75|             3.58|
|   1088|       42|3.369047619047619|
|  32460|        4|             4.25|
+-------+---------+-----------------+
only showing top 5 rows
Average rating and number of ratings per user (s

## Task 3 — Spark SQL

In [ ]:
ratings.createOrReplaceTempView("ratings_table")
movies.createOrReplaceTempView("movies_table")

**3.1 Simple SQL Queries**

*	Show first 20 rows
*	Count total ratings
*	Count distinct users
*	Average rating overall

**3.2 SQL Grouping**
*	Average rating by movie
*	Ratings count by movie
*	Best movies with at least 100 ratings
*	Worst movies with at least 100 ratings
*	Most active users

**3.3 SQL CASE WHEN**

Create rating buckets:

*	≥ 4.0 → “high”
*	3.0–3.9 → “medium”
*	< 3.0 → “low”

**3.4 SQL Window Functions**

*	Top 10 movies by rating for each genre
*	Earliest rating per user (row_number)
*	Rolling average rating per movie

In [ ]:
# Spark SQL solutions for Task 3

from pyspark.sql import functions as F

print("=== 3.1 Simple SQL Queries ===")
print("First 20 rows from ratings_table:")
spark.sql("SELECT * FROM ratings_table LIMIT 20").show()

print("Total number of ratings:")
spark.sql("SELECT COUNT(*) AS total_ratings FROM ratings_table").show()

print("Count of distinct users:")
spark.sql("SELECT COUNT(DISTINCT userId) AS distinct_users FROM ratings_table").show()

print("Average rating overall:")
spark.sql("SELECT AVG(rating) AS avg_rating FROM ratings_table").show()

print("=== 3.2 SQL Grouping ===")
print("Average rating and count by movie (with title, sample):")
spark.sql(
    """
    SELECT r.movieId, m.title,
           COUNT(*) AS n_ratings,
           AVG(r.rating) AS avg_rating
    FROM ratings_table r
    JOIN movies_table m ON r.movieId = m.movieId
    GROUP BY r.movieId, m.title
    """
).show(10, truncate=False)

print("Best movies with at least 100 ratings:")
spark.sql(
    """
    SELECT r.movieId, m.title,
           COUNT(*) AS n_ratings,
           AVG(r.rating) AS avg_rating
    FROM ratings_table r
    JOIN movies_table m ON r.movieId = m.movieId
    GROUP BY r.movieId, m.title
    HAVING COUNT(*) >= 100
    ORDER BY avg_rating DESC
    LIMIT 20
    """
).show(truncate=False)

print("Worst movies with at least 100 ratings:")
spark.sql(
    """
    SELECT r.movieId, m.title,
           COUNT(*) AS n_ratings,
           AVG(r.rating) AS avg_rating
    FROM ratings_table r
    JOIN movies_table m ON r.movieId = m.movieId
    GROUP BY r.movieId, m.title
    HAVING COUNT(*) >= 100
    ORDER BY avg_rating ASC
    LIMIT 20
    """
).show(truncate=False)

print("Most active users:")
spark.sql(
    """
    SELECT userId,
           COUNT(*) AS n_ratings,
           AVG(rating) AS avg_rating
    FROM ratings_table
    GROUP BY userId
    ORDER BY n_ratings DESC
    LIMIT 20
    """
).show()

print("=== 3.3 SQL CASE WHEN (rating buckets) ===")
print("Sample of rating buckets:")
spark.sql(
    """
    SELECT userId, movieId, rating,
           CASE
               WHEN rating >= 4.0 THEN 'high'
               WHEN rating >= 3.0 THEN 'medium'
               ELSE 'low'
           END AS rating_bucket
    FROM ratings_table
    LIMIT 20
    """
).show()

print("Bucket distribution:")
spark.sql(
    """
    SELECT CASE
               WHEN rating >= 4.0 THEN 'high'
               WHEN rating >= 3.0 THEN 'medium'
               ELSE 'low'
           END AS rating_bucket,
           COUNT(*) AS n_ratings
    FROM ratings_table
    GROUP BY rating_bucket
    ORDER BY n_ratings DESC
    """
).show()

print("=== 3.4 SQL Window Functions ===")
print("Top 10 movies by rating for each genre:")
spark.sql(
    """
    WITH exploded AS (
        SELECT r.userId, r.movieId, r.rating, m.title,
               explode(split(m.genres, '\\|')) AS genre
        FROM ratings_table r
        JOIN movies_table m ON r.movieId = m.movieId
    ),
    movie_genre_stats AS (
        SELECT movieId, title, genre,
               COUNT(*) AS n_ratings,
               AVG(rating) AS avg_rating
        FROM exploded
        GROUP BY movieId, title, genre
    ),
    ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY genre ORDER BY avg_rating DESC, n_ratings DESC) AS rank_in_genre
        FROM movie_genre_stats
    )
    SELECT genre, rank_in_genre, title, avg_rating, n_ratings
    FROM ranked
    WHERE rank_in_genre <= 10
    ORDER BY genre, rank_in_genre
    """
).show(truncate=False)

print("Earliest rating per user (using row_number):")
spark.sql(
    """
    SELECT userId, movieId, rating, timestamp
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY userId ORDER BY timestamp ASC) AS rn
        FROM ratings_table
    )
    WHERE rn = 1
    """
).show(20)

print("Rolling average rating per movie:")
spark.sql(
    """
    SELECT movieId, userId, rating, timestamp,
           AVG(rating) OVER (
               PARTITION BY movieId
               ORDER BY timestamp
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) AS rolling_avg_rating
    FROM ratings_table
    ORDER BY movieId, timestamp
    """
).show(20)

=== 3.1 Simple SQL Queries ===
First 20 rows from ratings_table:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|     70|   3.0|964982400|
|     1|    101|   5.0|964980868|
|     1|    110|   4.0|964982176|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
|     1|    163|   5.0|964983650|
|     1|    216|   5.0|964981208|
|     1|    223|   3.0|964980985|
|     1|    231|   5.0|964981179|
|     1|    235|   4.0|964980908|
|     1|    260|   5.0|964981680|
|     1|    296|   3.0|964982967|
|     1|    316|   3.0|964982310|
|     1|    333|   5.0|964981179|
|     1|    349|   4.0|964982563|
+------+-------+------+---------+

Total number of ratings:
+-------------+
|total_ratings|
+-------------+
|       100836|
+-------------+

Count of dis

## Task 4 — Joins & Genre Analytics

**4.1 Basic join**

Join ratings → movies on movieId.

*	Show 20 joined rows
*	Show userId, title, rating
*	Count how many ratings each genre has

**4.2 Parse genres**

genres looks like "Action|Adventure|Sci-Fi"

*	split genres into an array
*	explode the genres
*	count ratings per genre
*	compute average rating per genre

**4.3 Join + Aggregation**

Compute:

*	average rating per genre
*	average rating per movie title
*	number of ratings per movie
*	number of ratings per genre per year (bonus: extract year from title)

**4.4 Left Anti Join**

Find:

*	movies in movies.csv with no ratings
*	number of such movies

In [ ]:
from pyspark.sql.functions import col, split, explode, avg, count, regexp_extract

# === 4.1 Basic join ===
print("=== 4.1 Basic join ===")
ratings_movies = ratings.join(movies, "movieId")
print("Joined ratings and movies (20 rows):")
ratings_movies.show(20, truncate=False)

print("userId, title, rating (sample):")
ratings_movies.select("userId", "title", "rating").show(20, truncate=False)

# === 4.2 Parse genres ===
print("=== 4.2 Parse genres ===")
ratings_with_genres = (
    ratings_movies
    .withColumn("genre_array", split(col("genres"), "\\|"))
    .withColumn("genre", explode(col("genre_array")))
)

print("Ratings exploded by genre (sample):")
ratings_with_genres.select("userId", "movieId", "title", "genre", "rating").show(20, truncate=False)

genre_stats = ratings_with_genres.groupBy("genre").agg(
    count("*").alias("n_ratings"),
    avg("rating").alias("avg_rating"),
)
print("Ratings count and average per genre:")
genre_stats.orderBy(col("n_ratings").desc()).show(truncate=False)

# === 4.3 Join + Aggregation ===
print("=== 4.3 Join + Aggregation ===")
print("Average rating per genre:")
genre_stats.orderBy(col("avg_rating").desc()).show(truncate=False)

print("Average rating and number of ratings per movie title (sample):")
movie_title_stats = ratings_with_genres.groupBy("movieId", "title").agg(
    count("*").alias("n_ratings"),
    avg("rating").alias("avg_rating"),
)
movie_title_stats.orderBy(col("n_ratings").desc()).show(20, truncate=False)

print("Number of ratings per genre per year:")
ratings_with_year_genre = ratings_with_genres.withColumn(
    "year", regexp_extract(col("title"), "\\((\\d{4})\\)", 1)
)
genre_year_stats = ratings_with_year_genre.groupBy("genre", "year").agg(
    count("*").alias("n_ratings"),
    avg("rating").alias("avg_rating"),
)
genre_year_stats.orderBy("genre", "year").show(50, truncate=False)

# === 4.4 Left Anti Join ===
print("=== 4.4 Left Anti Join ===")
movies_without_ratings = movies.join(ratings, "movieId", "left_anti")
print("Movies with no ratings (sample):")
movies_without_ratings.show(20, truncate=False)
print("Number of movies with no ratings:", movies_without_ratings.count())

=== 4.1 Basic join ===
Joined ratings and movies (20 rows):
+-------+------+------+---------+-----------------------------------------+-------------------------------------------+
|movieId|userId|rating|timestamp|title                                    |genres                                     |
+-------+------+------+---------+-----------------------------------------+-------------------------------------------+
|1      |1     |4.0   |964982703|Toy Story (1995)                         |Adventure|Animation|Children|Comedy|Fantasy|
|3      |1     |4.0   |964981247|Grumpier Old Men (1995)                  |Comedy|Romance                             |
|6      |1     |4.0   |964982224|Heat (1995)                              |Action|Crime|Thriller                      |
|47     |1     |5.0   |964983815|Seven (a.k.a. Se7en) (1995)              |Mystery|Thriller                           |
|50     |1     |5.0   |964982931|Usual Suspects, The (1995)               |Crime|Mystery|Thriller   

## Task 5 — MLlib Classification

**Goal: Build a binary classifier:**

Predict whether a user will give rating ≥ 4.0

5.1 Create label

Add a column:

In [ ]:
from pyspark.sql.functions import when, col

# 5.1 Create binary label: 1 if rating >= 4, else 0
ratings_with_label = ratings.withColumn(
    "label",
    when(col("rating") >= 4.0, 1.0).otherwise(0.0)
)

print("Sample with label column:")
ratings_with_label.select("userId", "movieId", "rating", "label").show(10)

Sample with label column:
+------+-------+------+-----+
|userId|movieId|rating|label|
+------+-------+------+-----+
|     1|      1|   4.0|  1.0|
|     1|      3|   4.0|  1.0|
|     1|      6|   4.0|  1.0|
|     1|     47|   5.0|  1.0|
|     1|     50|   5.0|  1.0|
|     1|     70|   3.0|  0.0|
|     1|    101|   5.0|  1.0|
|     1|    110|   4.0|  1.0|
|     1|    151|   5.0|  1.0|
|     1|    157|   5.0|  1.0|
+------+-------+------+-----+
only showing top 10 rows


**5.2 Feature engineering**

Create features:

	•	rating (as-is)
	•	timestamp
	•	normalized timestamp
	•	(optional) number of ratings by that user (using join or window)

Use VectorAssembler to pack them.

**5.3 Split data**

70% train, 30% test.

**5.4 Train model**

Train a Logistic Regression model.

	•	print coefficients
	•	print intercept
	•	print ROC AUC

**5.5 Evaluate**

Compute :

	•	accuracy
	•	precision
	•	recall
	•	confusion matrix (TP, FP, TN, FN)

**5.6 Task: Improve model**

try:

	•	Adding a log-transformed timestamp
	•	Using a DecisionTreeClassifier
	•	Using a RandomForestClassifier
	•	Comparing metrics

In [ ]:
# MLlib Classification – Task 5

from pyspark.sql.functions import col, min as min_f, max as max_f, log, lit
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from time import time

print("=== 5.2 Feature engineering ===")

try:
    df = ratings_with_label
except NameError as e

# Add number of ratings by user as a feature
user_counts = df.groupBy("userId").count().withColumnRenamed("count", "user_rating_count")
df = df.join(user_counts, "userId")

# Compute normalized timestamp and log-transformed timestamp
ts_stats = df.agg(
    min_f("timestamp").alias("min_ts"),
    max_f("timestamp").alias("max_ts")
).collect()[0]

min_ts = ts_stats["min_ts"]
max_ts = ts_stats["max_ts"]
ts_range = max_ts - min_ts if max_ts != min_ts else 1

df = df.withColumn(
    "norm_timestamp", (col("timestamp") - lit(min_ts)) / lit(ts_range)
).withColumn(
    "log_timestamp", log(col("timestamp") + lit(1.0))
)

feature_cols = ["rating", "timestamp", "norm_timestamp", "user_rating_count", "log_timestamp"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

data_with_features = assembler.transform(df).select("features", "label")

print("Sample of features and label:")
data_with_features.show(10, truncate=False)

print("=== 5.3 Train/test split ===")
train_df, test_df = data_with_features.randomSplit([0.7, 0.3], seed=42)
print("Train count:", train_df.count(), "Test count:", test_df.count())

print("=== 5.4 Train Logistic Regression model ===")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
lr_model = lr.fit(train_df)

print("Coefficients:", lr_model.coefficients)
print("Intercept:", lr_model.intercept)

print("=== 5.5 Evaluate Logistic Regression ===")
lr_predictions = lr_model.transform(test_df)

lr_predictions.select("label", "prediction", "probability").show(10, truncate=False)

binary_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
roc_auc = binary_eval.evaluate(lr_predictions)
print("Logistic Regression ROC AUC:", roc_auc)

# Confusion matrix
TP = lr_predictions.filter("label = 1.0 AND prediction = 1.0").count()
TN = lr_predictions.filter("label = 0.0 AND prediction = 0.0").count()
FP = lr_predictions.filter("label = 0.0 AND prediction = 1.0").count()
FN = lr_predictions.filter("label = 1.0 AND prediction = 0.0").count()

total = TP + TN + FP + FN
accuracy = (TP + TN) / float(total) if total > 0 else 0.0
precision = TP / float(TP + FP) if (TP + FP) > 0 else 0.0
recall = TP / float(TP + FN) if (TP + FN) > 0 else 0.0

print(f"TP={TP}, FP={FP}, TN={TN}, FN={FN}")
print(f"Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}")

print("=== 5.6 Improve model: Decision Tree and Random Forest ===")

# Decision Tree
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5)
dt_model = dt.fit(train_df)
dt_predictions = dt_model.transform(test_df)
dt_auc = binary_eval.evaluate(dt_predictions)
print("Decision Tree ROC AUC:", dt_auc)

# Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=10)
rf_model = rf.fit(train_df)
rf_predictions = rf_model.transform(test_df)
rf_auc = binary_eval.evaluate(rf_predictions)
print("Random Forest ROC AUC:", rf_auc)

=== 5.2 Feature engineering ===
Sample of features and label:
+---------------------------------------------------------------+-----+
|features                                                       |label|
+---------------------------------------------------------------+-----+
|[4.0,9.64982703E8,0.19284624425107147,232.0,20.687620735826574]|1.0  |
|[4.0,9.64981247E8,0.19284419260665842,232.0,20.6876192269901]  |1.0  |
|[4.0,9.64982224E8,0.1928455692938779,232.0,20.687620239444495] |1.0  |
|[5.0,9.64983815E8,0.19284781116631003,232.0,20.68762188817817] |1.0  |
|[5.0,9.64982931E8,0.19284656552505924,232.0,20.68762097210021] |1.0  |
|[3.0,9.649824E8,0.1928458172948509,232.0,20.68762042183126]    |0.0  |
|[5.0,9.64980868E8,0.19284365855910857,232.0,20.68761883423628] |1.0  |
|[4.0,9.64982176E8,0.1928455016572489,232.0,20.687620189702645] |1.0  |
|[5.0,9.64984041E8,0.19284812962210493,232.0,20.68762212237896] |1.0  |
|[5.0,9.649841E8,0.19284821275879474,232.0,20.687622183519867]  |1.0  |
+-

## Task 6 — Performance & Execution

**6.1 Check partitions**

Get number of partitions for ratings and movies.

**6.2 Repartition and coalesce**

	•	explain difference
	•	show effect on shuffles
	•	check number of partitions with rdd.getNumPartitions()

**6.3 Cache**

Cache ratings:

	•	compute count
	•	compute average rating per movie twice
	•	measure difference using Python time

**6.4 explain(True)**

run explain() on:

	•	a join
	•	a groupBy
	•	a window function

And identify shuffle boundaries.

In [ ]:
from pyspark.sql.functions import avg
from pyspark.sql.window import Window
from time import time

# === 6.1 Check partitions ===
print("=== 6.1 Check partitions ===")
print("ratings partitions:", ratings.rdd.getNumPartitions())
print("movies partitions:", movies.rdd.getNumPartitions())

# === 6.2 Repartition and coalesce ===
print("=== 6.2 Repartition and coalesce ===")
print("Original ratings partitions:", ratings.rdd.getNumPartitions())

ratings_repart = ratings.repartition(8)
print("After repartition(8):", ratings_repart.rdd.getNumPartitions())

ratings_coalesced = ratings_repart.coalesce(4)
print("After coalesce(4):", ratings_coalesced.rdd.getNumPartitions())

print("Repartition shuffles data across the cluster, while coalesce reduces the number of partitions with minimal shuffling.")

print("Physical plan for repartition:")
ratings_repart.explain(True)
print("Physical plan for coalesce:")
ratings_coalesced.explain(True)

# === 6.3 Cache ===
print("=== 6.3 Cache ===")
ratings_cached = ratings.repartition(8).cache()

t0 = time()
ratings_cached.count()
t1 = time()
print(f"Count on cached DataFrame (first time): {t1 - t0:.4f} seconds")

t2 = time()
ratings_cached.count()
t3 = time()
print(f"Count on cached DataFrame (second time): {t3 - t2:.4f} seconds")

t4 = time()
avg_per_movie_1 = ratings_cached.groupBy("movieId").agg(avg("rating").alias("avg_rating"))
avg_per_movie_1.count()
t5 = time()
print(f"Average rating per movie (first computation): {t5 - t4:.4f} seconds")

t6 = time()
avg_per_movie_2 = ratings_cached.groupBy("movieId").agg(avg("rating").alias("avg_rating"))
avg_per_movie_2.count()
t7 = time()
print(f"Average rating per movie (second computation): {t7 - t6:.4f} seconds")

# === 6.4 explain(True) ===
print("=== 6.4 explain(True) ===")

join_df = ratings.join(movies, "movieId")
print("Explain join:")
join_df.explain(True)

group_df = ratings.groupBy("movieId").agg(avg("rating").alias("avg_rating"))
print("Explain groupBy:")
group_df.explain(True)

w_movie = Window.partitionBy("movieId").orderBy("timestamp")
window_df = ratings.withColumn("avg_movie_rating", avg("rating").over(w_movie))
print("Explain window function:")
window_df.explain(True)

=== 6.1 Check partitions ===
ratings partitions: 1
movies partitions: 1
=== 6.2 Repartition and coalesce ===
Original ratings partitions: 1
After repartition(8): 8
After coalesce(4): 4
Repartition shuffles data across the cluster, while coalesce reduces the number of partitions with minimal shuffling.
Physical plan for repartition:
== Parsed Logical Plan ==
Repartition 8, true
+- Relation [userId#1717,movieId#1718,rating#1719,timestamp#1720] csv

== Analyzed Logical Plan ==
userId: int, movieId: int, rating: double, timestamp: int
Repartition 8, true
+- Relation [userId#1717,movieId#1718,rating#1719,timestamp#1720] csv

== Optimized Logical Plan ==
Repartition 8, true
+- Relation [userId#1717,movieId#1718,rating#1719,timestamp#1720] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=18890]
         +- FileScan csv [userId#1717,movieId#17

## Stop Spark

In [ ]:
spark.stop()